# PaddleOCR-VL on Dr.DocBench pages
Settings: **Accelerator = GPU (T4 x2 or P100)**, **Internet = On**. Add the uploaded dataset as input.
Run top to bottom with `LIMIT = 20` first (smoke test), check the printed outputs, then set `LIMIT = None` and run again. The run is resumable (finished pages are skipped).

In [ ]:
!nvidia-smi | head -15
import sys; print(sys.version)

In [ ]:
import os, glob, csv, zipfile, shutil, json, time, sys, subprocess
LIMIT = 20            # smoke test: run this many pages spread over dev+test. Set to None for the full run (1495 pages).
SPLITS = ("dev", "test")

manifests = glob.glob("/kaggle/input/**/manifest.csv", recursive=True)      # Kaggle mounts datasets at /kaggle/input/datasets/<owner>/<slug>/
zips = glob.glob("/kaggle/input/**/drdocbench_pages.zip", recursive=True)
if manifests:
    root = os.path.dirname(manifests[0])
elif zips:
    zipfile.ZipFile(zips[0]).extractall("/kaggle/working/data"); root = "/kaggle/working/data"
else:
    root = None
assert root, "dataset not found under /kaggle/input: " + str(glob.glob("/kaggle/input/**", recursive=False))
rows = [r for r in csv.DictReader(open(root + "/manifest.csv")) if r["split"] in SPLITS]
if LIMIT:
    step = max(1, len(rows) // LIMIT); rows = rows[::step][:LIMIT]
stem = lambda r: f"{r['doc']}_{r['page']}"
print(len(rows), "pages to process; first:", rows[0]["file"])

In [ ]:
# PaddlePaddle replaces an NCCL library that Kaggle's preinstalled PyTorch needs, and PaddleOCR's dependencies import torch,
# which then crashes with "undefined symbol: ncclCommShrink". This notebook never uses torch, so remove it first.
!pip uninstall -y -q torch torchvision torchaudio
# The paddlepaddle.org.cn host is slow and sometimes times out (pip's default timeout is 15 s): long timeout, retries, and an outer retry loop.
!bash -c "for i in 1 2 3 4; do python -m pip install -q --timeout 180 --retries 10 paddlepaddle-gpu==3.2.1 -i https://www.paddlepaddle.org.cn/packages/stable/cu126/ && break; echo install attempt $i failed; sleep 30; done"
!bash -c "for i in 1 2 3; do python -m pip install -q --timeout 180 --retries 10 -U 'paddleocr[doc-parser]' && break; echo install attempt $i failed; sleep 30; done"
import os
os.environ["PADDLE_PDX_MODEL_SOURCE"] = "huggingface"          # fetch model weights from Hugging Face only
os.environ["PADDLE_PDX_DISABLE_MODEL_SOURCE_CHECK"] = "True"   # skip the hub connectivity probe
import paddle; print("paddle", paddle.__version__, "| gpu:", paddle.device.is_compiled_with_cuda())
import paddleocr; print("paddleocr", paddleocr.__version__)

In [ ]:
OUT = "/kaggle/working/paddleocrvl_out"; os.makedirs(OUT, exist_ok=True)
todo = [r for r in rows if not os.path.exists(f"{OUT}/{stem(r)}/done")]
print(len(todo), "pages left")
json.dump([{"file": f"{root}/{r['file']}", "stem": stem(r)} for r in todo], open("/kaggle/working/rows.json", "w"))

WORKER = """
import os, sys, json, time
shard, nshards = int(sys.argv[1]), int(sys.argv[2])
os.environ["CUDA_VISIBLE_DEVICES"] = str(shard)          # one worker per GPU; must be set before paddle is imported
os.environ["PADDLE_PDX_MODEL_SOURCE"] = "huggingface"
os.environ["PADDLE_PDX_DISABLE_MODEL_SOURCE_CHECK"] = "True"
from paddleocr import PaddleOCRVL
OUT = "/kaggle/working/paddleocrvl_out"
items = json.load(open("/kaggle/working/rows.json"))[shard::nshards]
pipe = PaddleOCRVL()
def save(it, res):
    d = f"{OUT}/{it['stem']}"; os.makedirs(d, exist_ok=True)
    res.save_to_json(save_path=d); res.save_to_markdown(save_path=d); open(d + "/done", "w").close()
t0 = time.time(); fails = []
for i in range(0, len(items), 8):
    batch = items[i:i + 8]
    try:
        results = list(pipe.predict([b["file"] for b in batch])); assert len(results) == len(batch)
        for it, res in zip(batch, results): save(it, res)
    except Exception as e:                                # one bad page must not sink a batch
        print("batch failed, retrying singly:", repr(e)[:200], flush=True)
        for it in batch:
            try: save(it, list(pipe.predict(it["file"]))[0])
            except Exception as e2: fails.append((it["stem"], repr(e2)[:200]))
    n = min(i + 8, len(items)); el = time.time() - t0
    print(f"shard {shard}: {n}/{len(items)}  {el:.0f}s ({el / n:.1f}s/page) failures {len(fails)}", flush=True)
json.dump(fails, open(f"{OUT}/_failures_{shard}.json", "w"))
"""
open("/kaggle/working/worker.py", "w").write(WORKER)
import paddle; NGPU = max(1, paddle.device.cuda.device_count()); print("GPUs:", NGPU)
procs = [subprocess.Popen([sys.executable, "/kaggle/working/worker.py", str(k), str(NGPU)],
                          stdout=open(f"/kaggle/working/worker{k}.log", "w"), stderr=subprocess.STDOUT) for k in range(NGPU)]
while any(p.poll() is None for p in procs):
    time.sleep(120)
    for k in range(NGPU): print(open(f"/kaggle/working/worker{k}.log").read().strip().split("\n")[-1][:170])
print("exit codes:", [p.returncode for p in procs])

In [ ]:
# what did it produce? (paste this output back to me after the smoke run)
import itertools
d = sorted(glob.glob(OUT + "/*/"))[0]
print(d); print(sorted(os.listdir(d)))
j = [f for f in os.listdir(d) if f.endswith(".json")][0]
data = json.load(open(d + j))
print("top-level keys:", list(data.keys()))
blocks = data.get("parsing_res_list", [])
print(len(blocks), "layout blocks; labels:", sorted({b.get("block_label") for b in blocks}))
print({k: (v if k != "block_content" else v[:60]) for k, v in blocks[0].items()} if blocks else None)

In [ ]:
shutil.make_archive("/kaggle/working/paddleocrvl_results", "zip", OUT)
print("wrote /kaggle/working/paddleocrvl_results.zip", round(os.path.getsize("/kaggle/working/paddleocrvl_results.zip") / 1e6, 1), "MB  -> download it from the Output tab")